# 中研院 strength estimator 推論

`aicup/strength_se.py` 輸出每局目標玩家各手的 SE∞ `score`（每種對稱各一組平均與標準差）與 SL 11 級機率的平均。

- 這份 notebook 是實驗，不在競賽一最終版本流程內。
- 外部資源：rlglab/strength-estimator 的 `go.tar.gz`（GPL-3.0），由程式從官方網址下載，只用 `se_infty_go.pt`、`sl_go.pt`
- CSV 先複製到 `/content`，推論期間不讀 Drive；結果寫在 `/content`，每批結束後複製到 Drive
- 斷線時最多損失最近一批，重新開 runtime 依序執行全部 cell 就會接續；推論中不要中斷 cell

| MODE | 內容 |
|---|---|
| `val` | 和 `val648-p3.jsonl` 相同的 12,960 局，預設 8 種對稱 |
| `val_opp` | 同樣的 12,960 局，改評對手的手，預設 4 種對稱 |
| `test` | 競賽二的測試題，沒有預設對稱數，要依 val 的 stacking 結果手動設定 `SYM`；dtype 沿用 val 的輸出 |

- `SYM`：`None` 時用 MODE 的預設對稱數。
- `BF16_CHECK`：只用於 `val`。先用前 64 局比對 bf16 與 fp32，通過才採用 bf16。

## 需要的檔案

路徑都在 Drive 的 `MyDrive/aicup-2026/` 下。

| 路徑 | MODE | 產生方式 |
|---|---|---|
| `training-v1.1/training/train_*.csv`（10 個） | 全部 | 官方訓練集 v1.1，見 README |
| `humansl-analysis/val648-p3.jsonl` | 全部 | `colab_humansl.ipynb` 的 `val648` |
| `humansl-analysis/val648-se.jsonl` | `val_opp`、`test` | 這份 notebook 的 `val` |
| `tests/rank_prediction_test_{public,private}.csv` | `test` | AI CUP 競賽平台 |

## 參數

In [ ]:
REPO_URL = 'https://github.com/Jx-study/aicup-2026-go.git'
COMMIT = 'origin/main'  # 重現特定版本時填 commit hash
MODE = 'val'
SYM = None
BATCH = 1000
BF16_CHECK = True
AUTO_UNASSIGN = True

RUNS = {  # MODE: (輸入參數, 輸出檔名, 總局數, 預設對稱數)
    'val': (['--keys-from', '/content/humansl-analysis/val648-p3.jsonl'], 'val648-se.jsonl', 12960, 8),
    'val_opp': (['--keys-from', '/content/humansl-analysis/val648-p3.jsonl', '--side', 'opponent'],
                'val648-se-opp.jsonl', 12960, 4),
    'test': (['--test-dir', '/content/tests'], 'test-se.jsonl', 10000, None),
}
if MODE not in RUNS:
    raise SystemExit('MODE 要是 %s 之一' % ', '.join(RUNS))
SRC, OUT_NAME, TOTAL, SYM_DEFAULT = RUNS[MODE]
SYM = SYM or SYM_DEFAULT
assert SYM, 'test 模式要手動設定 SYM（val 的 stacking 選定的對稱數）'
COMMON = ['--csv', '/content/training-v1.1/training/train_*.csv', '--model-dir', '/content/se-models']
NEEDS = [('training-v1.1/training', '官方訓練集 v1.1，見 README'),
         ('humansl-analysis/val648-p3.jsonl', 'colab_humansl.ipynb 的 MODE val648')]
if MODE != 'val':
    NEEDS.append(('humansl-analysis/val648-se.jsonl', '這份 notebook 的 MODE val'))
if MODE == 'test':
    NEEDS += [('tests/rank_prediction_test_%s.csv' % s, 'AI CUP 競賽平台') for s in ('public', 'private')]
print('%s：輸出 %s，%d 種對稱' % (MODE, OUT_NAME, SYM))
print(' '.join(['python', '-m', 'aicup.strength_se', *SRC, *COMMON, '--symmetries', str(SYM)]))

## 取得程式

要看到：Drive 掛載完成、最新的 commit、GPU 型號。

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sgfmill'], check=True)
from aicup.colab_nb import DRIVE_ROOT, check_commit, link_drive, need, run

GIT_HEAD = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()

!git log --oneline -1
!nvidia-smi -L

## 準備資料

要看到：需要的檔案都在、複製的清單。Drive 上已有結果時會複製回 `/content`，接續推論。

In [ ]:
import shutil

need([(os.path.join(DRIVE_ROOT, rel), how) for rel, how in NEEDS])
for rel, _ in NEEDS:
    src, dst = os.path.join(DRIVE_ROOT, rel), '/content/' + rel
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(src, dst)
    print('複製', rel)
LOCAL = '/content/' + OUT_NAME
BACKUP = DRIVE_ROOT + '/humansl-analysis'
if os.path.exists(BACKUP + '/' + OUT_NAME) and not os.path.exists(LOCAL):
    shutil.copy(BACKUP + '/' + OUT_NAME, LOCAL)
    print('接續', OUT_NAME)

## 冒煙測試與 dtype

每個 MODE 都先用前 64 局冒煙測試，第一次執行會下載 `go.tar.gz`，最後印出速度、估計時間與推論要用的參數。

- `val`：先跑 fp32。`BF16_CHECK` 時再跑 bf16 比對：每局 `score_mean` 的最大差小於 0.05、`sl_rank` 的最大差小於 0.005 才採用 bf16。
- `val_opp`、`test`：不比對，dtype 沿用 `val648-se.jsonl`，各 MODE 的設定才會一致。

估計時間超過 3 小時時先停，決定是否減少對稱數。

In [ ]:
import glob, json, re, shlex
import numpy as np


def run_se(args, out):
    argstr = shlex.join(args)  # glob 要原樣傳給程式，不讓 shell 展開
    lines = !python -m aicup.strength_se {argstr} --out {out} 2>&1
    lines = [line for line in lines if 'warn' not in line.lower()]
    print('\n'.join(lines[-3:]))
    speed = [float(m.group(1)) for line in lines for m in [re.search(r'每秒 (\d+) 個局面', line)] if m]
    if not speed:
        raise SystemExit('沒有跑完，看上面的錯誤訊息')
    return speed[-1]


def rows(path):
    return {r['key']: r for r in map(json.loads, open(path))}


SMOKE = [*SRC, *COMMON, '--symmetries', str(SYM), '--limit', '64']
for p in glob.glob('/content/smoke-*.jsonl'):
    os.remove(p)
if MODE == 'val':
    DTYPE = 'float32'
    speed = run_se(SMOKE, '/content/smoke-fp32.jsonl')
    a = rows('/content/smoke-fp32.jsonl')
    if BF16_CHECK:
        speed16 = run_se(SMOKE + ['--dtype', 'bfloat16'], '/content/smoke-bf16.jsonl')
        b = rows('/content/smoke-bf16.jsonl')
        ds = max(np.abs(np.subtract(a[k]['score_mean'], b[k]['score_mean'])).max() for k in a)
        dr = max(np.abs(np.subtract(a[k]['sl_rank'], b[k]['sl_rank'])).max() for k in a)
        print('bf16 對 fp32：%d 局，score_mean 最大差 %.4f，sl_rank 最大差 %.4f，速度 %.0f 對 %.0f'
              % (len(a), ds, dr, speed16, speed))
        if len(b) == len(a) and ds < 0.05 and dr < 0.005:
            DTYPE, speed = 'bfloat16', speed16
            print('採用 bf16')
        else:
            print('不採用 bf16，維持 float32')
else:
    DTYPE = json.loads(open('/content/humansl-analysis/val648-se.jsonl').readline())['dtype']
    print('沿用 val 的 dtype：%s' % DTYPE)
    speed = run_se(SMOKE + ['--dtype', DTYPE], '/content/smoke-%s.jsonl' % DTYPE)
    a = rows('/content/smoke-%s.jsonl' % DTYPE)
per_game = np.mean([r['n'] for r in a.values()])
print('估計 %.1f 小時（每局 %.0f 個局面，每秒 %.0f 個局面）' % (TOTAL * per_game / speed / 3600, per_game, speed))
CMD = ['python', '-m', 'aicup.strength_se', *SRC, *COMMON, '--dtype', DTYPE, '--symmetries', str(SYM)]
print(' '.join(CMD))

## 執行

每批結束會印出 Drive 上的行數，數字增加代表備份成功。全部跑完時行數等於總局數。

In [ ]:
LOG = '/content/%s-se.log' % MODE
os.makedirs(BACKUP, exist_ok=True)
for n in sorted({*range(BATCH, TOTAL, BATCH), TOTAL}):
    run([*CMD, '--limit', str(n), '--out', LOCAL], LOG)
    shutil.copy(LOCAL, BACKUP)
    shutil.copy(LOG, BACKUP)
    print('Drive 上 %d 行' % sum(1 for _ in open(BACKUP + '/' + OUT_NAME)))

## 檢查

要看到：行數等於總局數、沒有重複 key、dtype 與對稱數符合設定。

In [ ]:
out_rows = [json.loads(l) for l in open(LOCAL)]
keys = [r['key'] for r in out_rows]
print('已完成 %d / %d 局，重複 key %d' % (len(out_rows), TOTAL, len(keys) - len(set(keys))))
print('dtype：', sorted({r.get('dtype') for r in out_rows}))

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣運算單元。

In [ ]:
from google.colab import runtime

drive.flush_and_unmount()  # 等 Drive 寫入完成
if AUTO_UNASSIGN:
    runtime.unassign()
else:
    print('未釋放 runtime，用完記得手動中斷')